In [34]:
import pandas as pd
import numpy as np
import ast
import hdbscan

from scipy.interpolate import interp1d
from sklearn.decomposition import PCA

PATH_KEYPOINTS = "../Data/Unprocessed/keypoints.csv"
PATH_ACTIONS   = "../Data/Processed/actions_filtered.csv"

In [35]:
df_keypoints = pd.read_csv(PATH_KEYPOINTS)
df_actions = pd.read_csv(PATH_ACTIONS)

df_merged = df_keypoints.merge(df_actions, on=["file", "fencer"], how="left")
df_merged = df_merged[
    (df_merged["frame"] >= df_merged["start_frame"]) &
    (df_merged["frame"] <= df_merged["end_frame"])
]

In [36]:
def parse_keypoints(kpts):
    if kpts is None:
        return None

    # Already numeric
    if isinstance(kpts, (list, np.ndarray)):
        return np.asarray(kpts, dtype=np.float32)

    # String representation
    if isinstance(kpts, str):
        try:
            return np.asarray(ast.literal_eval(kpts), dtype=np.float32)
        except Exception:
            return None

    return None

def normalize_pose(kpts):
    kpts = np.asarray(kpts, dtype=np.float32)

    center = kpts.mean(axis=0)
    kpts = kpts - center

    scale = np.linalg.norm(kpts[5] - kpts[11]) + 1e-6
    return kpts / scale


def resample_action(poses, target_len=20):
    poses = np.stack(poses)  # (T, K, 2)
    T = poses.shape[0]

    t_old = np.linspace(0, 1, T)
    t_new = np.linspace(0, 1, target_len)

    poses_rs = np.zeros((target_len, poses.shape[1], 2))

    for k in range(poses.shape[1]):
        for d in range(2):
            f = interp1d(t_old, poses[:, k, d], kind="linear")
            poses_rs[:, k, d] = f(t_new)

    return poses_rs

In [37]:
X = []
action_ids = []
labels = []

for action_id, group in df_merged.groupby("action_id"):
    poses = []

    for kpts in group.sort_values("frame")["keypoints"]:
        kpts_parsed = parse_keypoints(kpts)
        if kpts_parsed is None or len(kpts_parsed) == 0:
            continue
        poses.append(normalize_pose(kpts_parsed))

    if len(poses) < 5:
        continue  # too short to be meaningful

    poses_rs = resample_action(poses, target_len=20)
    X.append(poses_rs.flatten())

    action_ids.append(action_id)
    labels.append(group["action"].iloc[0])  # original semantic label

X = np.vstack(X)

In [44]:
X_pca = PCA(
    n_components=30,
    whiten=True,
    random_state=42
).fit_transform(X)

clusterer = hdbscan.HDBSCAN(
    min_cluster_size=10,
    min_samples=3
)

clusters = clusterer.fit_predict(X_pca)
print(np.unique(clusters, return_counts=True))

cluster_df = pd.DataFrame({
    "action_id": action_ids,
    "cluster": clusters,
    "label": labels
})

pd.crosstab(
    cluster_df["cluster"],
    cluster_df["label"],
    normalize="index"
)

(array([-1,  0,  1]), array([304,  45,  51]))


label,ATTACK_BEAT,ATTACK_COUNTER,ATTACK_FLUNGE,ATTACK_LUNGE,ATTACK_REMISE,ATTACK_RIPOSTE,ATTACK_STEP_CUT,ATTACK_STOP_CUT,DEFENSE_DISTANCE_PULL,DEFENSE_PARRY,DEFENSE_POINT_IN_LINE
cluster,,,,,,,,,,,
-1,0.019737,0.039474,0.032895,0.467105,0.009868,0.029605,0.072368,0.052632,0.194079,0.049342,0.032895
0,0.155556,0.000000,0.000000,0.333333,0.022222,0.022222,0.066667,0.066667,0.000000,0.111111,0.222222
1,0.117647,0.078431,0.000000,0.098039,0.019608,0.019608,0.117647,0.274510,0.039216,0.019608,0.215686
